# Chapter 3: Finite Markov Decision Processes

A hands-on notebook for the core ideas in Chapter 3 of Sutton & Barto.

## Topics

- MDPs
- States, actions, rewards, and transitions
- Policies
- Returns and discounting
- State-value functions \(V^\pi(s)\)
- Action-value functions \(Q^\pi(s,a)\)
- Bellman expectation equations
- Optimal value functions
- Bellman optimality equations
- Deriving an optimal policy

The conceptual chain is

\[
\text{MDP}
\rightarrow
\text{Policy}
\rightarrow
\text{Value functions}
\rightarrow
\text{Bellman equations}
\rightarrow
\text{Optimal policy}.
\]


In [2]:
import numpy as np
N, g = 5, 0.9
A, Ap, B, Bp = (0,1), (4,1), (0,3), (2,3)
moves = [(-1,0), (1,0), (0,1), (0,-1)]
idx = lambda r, c: r*N + c

M = np.eye(N*N)          # will become I - gamma * P
b = np.zeros(N*N)        # expected immediate reward
for r in range(N):
    for c in range(N):
        s = idx(r, c)
        for dr, dc in moves:
            if (r, c) == A:   nr, nc, rew = *Ap, 10
            elif (r, c) == B: nr, nc, rew = *Bp, 5
            else:
                nr, nc = r+dr, c+dc
                if 0 <= nr < N and 0 <= nc < N: rew = 0
                else: nr, nc, rew = r, c, -1
            M[s, idx(nr, nc)] -= 0.25 * g
            b[s] += 0.25 * rew

v = np.linalg.solve(M, b).reshape(N, N)

print(v)

[[ 3.30899634  8.78929186  4.42761918  5.32236759  1.49217876]
 [ 1.52158807  2.99231786  2.25013995  1.9075717   0.54740271]
 [ 0.05082249  0.73817059  0.67311326  0.35818621 -0.40314114]
 [-0.9735923  -0.43549543 -0.35488227 -0.58560509 -1.18307508]
 [-1.85770055 -1.34523126 -1.22926726 -1.42291815 -1.97517905]]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Finite MDP

At time \(t\), the agent is in state \(S_t\), chooses action \(A_t\), and the environment returns reward \(R_{t+1}\) and next state \(S_{t+1}\).

\[
S_t \rightarrow A_t \rightarrow (R_{t+1}, S_{t+1})
\]

The environment dynamics are described by

\[
p(s',r\mid s,a).
\]

The Markov property means the current state contains the information needed to model the next transition.


## 2. Example environment

We use the same small deterministic MDP throughout the notebook.

States:

\[
S=\{A,B,C\}
\]

plus a terminal state.

Each non-terminal state has actions `left` and `right`.

```text
A --right/0--> B --right/0--> C --right/+10--> terminal
|               |              |
left/+1         left/-1        left/0
|               |              |
terminal        A              B
```

The interesting choice is at state \(A\): take an immediate \(+1\), or move toward a delayed \(+10\).


In [ ]:
states = ["A", "B", "C"]
actions = ["left", "right"]

# (next_state, reward, terminal)
transition = {
    ("A", "left"):  ("TERMINAL", 1.0, True),
    ("A", "right"): ("B",        0.0, False),
    ("B", "left"):  ("A",       -1.0, False),
    ("B", "right"): ("C",        0.0, False),
    ("C", "left"):  ("B",        0.0, False),
    ("C", "right"): ("TERMINAL", 10.0, True),
}

for key, value in transition.items():
    print(f"{key} -> {value}")


## 3. Policy

A stochastic policy specifies

\[
\pi(a\mid s)=P(A_t=a\mid S_t=s).
\]

For example, a random policy may choose both actions with probability \(0.5\).

A deterministic policy assigns one action with probability \(1\).


In [ ]:
random_policy = {
    "A": {"left": 0.5, "right": 0.5},
    "B": {"left": 0.5, "right": 0.5},
    "C": {"left": 0.5, "right": 0.5},
}

right_policy = {
    "A": {"left": 0.0, "right": 1.0},
    "B": {"left": 0.0, "right": 1.0},
    "C": {"left": 0.0, "right": 1.0},
}


## 4. Return and discounting

The discounted return is

\[
G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots
\]

where \(0\le\gamma\le1\).

- Small \(\gamma\): short-sighted
- Large \(\gamma\): long-sighted

A reward of \(10\) two steps in the future contributes

\[
10\gamma^2.
\]


In [ ]:
gammas = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99]

for gamma in gammas:
    value = 10 * gamma**2
    print(f"gamma={gamma:>4}: discounted future reward = {value:.3f}")


In [ ]:
steps = np.arange(0, 21)

plt.figure(figsize=(9, 5))
for gamma in [0.3, 0.5, 0.9, 0.99]:
    plt.plot(steps, gamma**steps, marker="o", label=f"gamma={gamma}")

plt.xlabel("Steps into the future")
plt.ylabel("Discount weight")
plt.title("Discount factor and temporal horizon")
plt.legend()
plt.show()


## 5. State-value function

The state-value function is

\[
V^\pi(s)
=
\mathbb{E}_\pi[G_t\mid S_t=s].
\]

It asks:

> If I start in state \(s\) and follow policy \(\pi\), how much return should I expect?

The policy matters. The same state can have very different values under different policies.


## 6. Action-value function

The action-value function is

\[
Q^\pi(s,a)
=
\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a].
\]

It asks:

> If I am in state \(s\), take action \(a\), and then follow policy \(\pi\), how much return should I expect?

The relationship between the two value functions is

\[
V^\pi(s)=\sum_a \pi(a\mid s)Q^\pi(s,a).
\]


## 7. Bellman expectation equation for \(V^\pi\)

The Bellman expectation equation is

\[
V^\pi(s)
=
\sum_a \pi(a\mid s)
\sum_{s',r}
p(s',r\mid s,a)
\left[
r+\gamma V^\pi(s')
\right].
\]

Its meaning is wonderfully compact:

\[
\boxed{\text{value now}=\text{reward now}+\text{discounted expected value later}}
\]

For our deterministic MDP, if the policy always goes right:

\[
V^\pi(C)=10,
\]

\[
V^\pi(B)=\gamma V^\pi(C)=10\gamma,
\]

and

\[
V^\pi(A)=\gamma V^\pi(B)=10\gamma^2.
\]


In [ ]:
def evaluate_policy(policy, gamma=0.9, theta=1e-10):
    V = {s: 0.0 for s in states}

    while True:
        delta = 0.0
        new_V = V.copy()

        for s in states:
            value = 0.0

            for a, prob in policy[s].items():
                next_state, reward, done = transition[(s, a)]
                future = 0.0 if done else V[next_state]
                value += prob * (reward + gamma * future)

            new_V[s] = value
            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V

        if delta < theta:
            break

    return V


In [ ]:
for gamma in [0.3, 0.5, 0.9, 0.99]:
    V = evaluate_policy(right_policy, gamma)
    print(f"gamma={gamma}: {V}")


## 8. Bellman expectation equation for \(Q^\pi\)

For action values,

\[
Q^\pi(s,a)
=
\sum_{s',r}
p(s',r\mid s,a)
\left[
r+
\gamma
\sum_{a'}
\pi(a'\mid s')
Q^\pi(s',a')
\right].
\]

The sequence is:

1. take action \(a\),
2. receive reward \(r\),
3. reach \(s'\),
4. continue following policy \(\pi\).


In [ ]:
def compute_q_from_v(V, gamma=0.9):
    Q = {}

    for s in states:
        for a in actions:
            next_state, reward, done = transition[(s, a)]
            future = 0.0 if done else V[next_state]
            Q[(s, a)] = reward + gamma * future

    return Q


V_random = evaluate_policy(random_policy, gamma=0.9)
Q_random = compute_q_from_v(V_random, gamma=0.9)

print("V under random policy:")
for s, v in V_random.items():
    print(s, round(v, 4))

print("\nQ under random policy:")
for key, value in Q_random.items():
    print(key, round(value, 4))


## 9. Verify \(V^\pi\) from \(Q^\pi\)

We should recover

\[
V^\pi(s)=\sum_a\pi(a\mid s)Q^\pi(s,a).
\]


In [ ]:
for s in states:
    reconstructed = sum(
        random_policy[s][a] * Q_random[(s, a)]
        for a in actions
    )

    print(
        f"{s}: V={V_random[s]:.6f}, "
        f"from Q={reconstructed:.6f}"
    )


## 10. Optimal value functions

The optimal state-value function is

\[
V_*(s)=\max_\pi V^\pi(s).
\]

The optimal action-value function is

\[
Q_*(s,a)=\max_\pi Q^\pi(s,a).
\]

If \(Q_*\) is known, an optimal action can be selected by

\[
\pi_*(s)=\arg\max_a Q_*(s,a).
\]


## 11. Bellman optimality equation

For state values:

\[
V_*(s)
=
\max_a
\sum_{s',r}
p(s',r\mid s,a)
\left[
r+\gamma V_*(s')
\right].
\]

Compare the two structures.

Bellman expectation equation:

\[
\sum_a \pi(a\mid s)(\cdots)
\]

Bellman optimality equation:

\[
\max_a(\cdots)
\]

That replacement changes the question from:

> How good is this policy?

to:

> What is the best achievable behavior?


For optimal action values:

\[
Q_*(s,a)
=
\sum_{s',r}
p(s',r\mid s,a)
\left[
r+\gamma\max_{a'}Q_*(s',a')
\right].
\]

This structure will later reappear in Q-learning.


## 12. Compute \(V_*\) with value iteration

Repeatedly apply the Bellman optimality backup:

\[
V(s)\leftarrow
\max_a
\left[
r+\gamma V(s')
\right].
\]

For a finite discounted MDP, this converges to \(V_*\).


In [ ]:
def value_iteration(gamma=0.9, theta=1e-10):
    V = {s: 0.0 for s in states}
    history = []

    while True:
        delta = 0.0
        new_V = V.copy()

        for s in states:
            action_values = []

            for a in actions:
                next_state, reward, done = transition[(s, a)]
                future = 0.0 if done else V[next_state]
                action_values.append(reward + gamma * future)

            new_V[s] = max(action_values)
            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V
        history.append(V.copy())

        if delta < theta:
            break

    return V, history


In [ ]:
V_star, history = value_iteration(gamma=0.9)

print("Optimal state values:")
for s, value in V_star.items():
    print(f"V*({s}) = {value:.4f}")

print("Iterations:", len(history))


In [ ]:
iterations = np.arange(1, len(history) + 1)

plt.figure(figsize=(9, 5))

for s in states:
    plt.plot(
        iterations,
        [h[s] for h in history],
        marker="o",
        label=f"State {s}"
    )

plt.xlabel("Iteration")
plt.ylabel("Value estimate")
plt.title("Value iteration convergence")
plt.legend()
plt.show()


## 13. Derive the optimal policy

After finding \(V_*\), compute

\[
Q_*(s,a)=r+\gamma V_*(s')
\]

for each action and select the largest.


In [ ]:
def greedy_policy_from_v(V, gamma=0.9):
    policy = {}
    Q = {}

    for s in states:
        action_values = {}

        for a in actions:
            next_state, reward, done = transition[(s, a)]
            future = 0.0 if done else V[next_state]
            q = reward + gamma * future

            action_values[a] = q
            Q[(s, a)] = q

        policy[s] = max(action_values, key=action_values.get)

    return policy, Q


optimal_policy, Q_star = greedy_policy_from_v(V_star, gamma=0.9)

print("Optimal policy:")
for s, a in optimal_policy.items():
    print(f"{s} -> {a}")

print("\nOptimal Q-values:")
for key, value in Q_star.items():
    print(key, round(value, 4))


## 14. How \(\gamma\) changes the optimal policy

At state \(A\):

- `left` gives \(+1\) immediately,
- `right` can eventually lead to \(+10\).

So changing \(\gamma\) can change the optimal action itself.

This is more important than merely changing the numerical value function.


In [ ]:
gamma_values = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]

for gamma in gamma_values:
    V_gamma, _ = value_iteration(gamma=gamma)
    policy_gamma, _ = greedy_policy_from_v(V_gamma, gamma=gamma)

    print(
        f"gamma={gamma:<4} "
        f"V*(A)={V_gamma['A']:.3f} "
        f"optimal action at A={policy_gamma['A']}"
    )


## 15. Policy evaluation vs. control

### Prediction / policy evaluation

Given policy \(\pi\), estimate

\[
V^\pi
\]

or

\[
Q^\pi.
\]

Question:

> How good is this policy?

### Control

Find

\[
\pi_*.
\]

Question:

> What should the agent do?

This distinction becomes central in later chapters.


## 16. Bellman backup intuition

Suppose:

\[
V(C)=10.
\]

If state \(B\) reaches \(C\) with reward \(0\) and \(\gamma=0.9\),

\[
V(B)=0+0.9(10)=9.
\]

If state \(A\) then reaches \(B\),

\[
V(A)=0+0.9(9)=8.1.
\]

So the distant terminal reward propagates backward:

```text
Terminal reward +10
       ↑
       C    value ≈ 10
       ↑
       B    value ≈ 9
       ↑
       A    value ≈ 8.1
```

That backward flow of information is one of the main ideas behind dynamic programming and later temporal-difference learning.


## 17. Plot \(V_*\) and \(Q_*\)


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(states, [V_star[s] for s in states])
plt.xlabel("State")
plt.ylabel("V*(s)")
plt.title("Optimal state values")
plt.show()


In [ ]:
labels = []
values = []

for s in states:
    for a in actions:
        labels.append(f"{s}-{a}")
        values.append(Q_star[(s, a)])

plt.figure(figsize=(9, 4))
plt.bar(labels, values)
plt.xlabel("State-action pair")
plt.ylabel("Q*(s,a)")
plt.title("Optimal action values")
plt.xticks(rotation=45)
plt.show()


## 18. Key equations

### Policy

\[
\pi(a\mid s)=P(A_t=a\mid S_t=s)
\]

### Return

\[
G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots
\]

### State value

\[
V^\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s]
\]

### Action value

\[
Q^\pi(s,a)=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a]
\]

### Relationship

\[
V^\pi(s)=\sum_a\pi(a\mid s)Q^\pi(s,a)
\]

### Bellman expectation equation

\[
V^\pi(s)
=
\sum_a\pi(a\mid s)
\sum_{s',r}
p(s',r\mid s,a)
[r+\gamma V^\pi(s')]
\]

### Bellman optimality equation

\[
V_*(s)
=
\max_a
\sum_{s',r}
p(s',r\mid s,a)
[r+\gamma V_*(s')]
\]

### Optimal policy

\[
\pi_*(s)=\arg\max_aQ_*(s,a).
\]


## 19. Exercise: state value from action values

Suppose

\[
Q^\pi(s,\text{left})=3,
\qquad
Q^\pi(s,\text{right})=7
\]

and

\[
\pi(\text{left}\mid s)=0.25,
\qquad
\pi(\text{right}\mid s)=0.75.
\]

Compute \(V^\pi(s)\) before running the cell.


In [ ]:
q_left = 3
q_right = 7

p_left = 0.25
p_right = 0.75

V = p_left * q_left + p_right * q_right
print("V^pi(s) =", V)


## 20. Exercise: one-step Bellman backup

Suppose

\[
R_{t+1}=2,
\qquad
V^\pi(S_{t+1})=8,
\qquad
\gamma=0.9.
\]

Compute

\[
R_{t+1}+\gamma V^\pi(S_{t+1}).
\]


In [ ]:
reward = 2
next_value = 8
gamma = 0.9

backup = reward + gamma * next_value
print("Bellman backup =", backup)


## 21. Experiment

Change the immediate reward:

```python
("A", "left"): ("TERMINAL", 1.0, True)
```

to

```python
("A", "left"): ("TERMINAL", 5.0, True)
```

Then recompute the optimal policy for

\[
\gamma=0.3,\;0.5,\;0.9,\;0.99.
\]

Watch when the agent changes from preferring immediate reward to delayed reward.


# 22. Chapter 3 mental model

```text
Environment dynamics
       ↓
     Policy π
       ↓
Expected future rewards
       ↓
  Vπ(s), Qπ(s,a)
       ↓
Bellman expectation equations
       ↓
Compare actions
       ↓
  V*(s), Q*(s,a)
       ↓
Bellman optimality equations
       ↓
 Optimal policy π*
```

The central idea is:

\[
\boxed{\text{Value functions compress future consequences into useful numbers now.}}
\]

Bellman equations exploit recursion:

\[
\boxed{\text{value now}=\text{reward now}+\text{value later}.}
\]

Optimal control changes averaging under a policy into maximization over actions.
